In [2]:
import numpy as np
import pandas as pd

# Load cars.csv directly from the CampusX GitHub repository
url = "https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/main/day27-one-hot-encoding/cars.csv"
df = pd.read_csv(url)

print(df.head())

# Check unique category counts across columns
print("Unique Brands:", df['brand'].nunique()) # 32 brands
print("\nBrand Counts:\n", df['brand'].value_counts())
print("\nFuel Counts:\n", df['fuel'].value_counts())   # 4 categories
print("\nOwner Counts:\n", df['owner'].value_counts()) # 5 categories

     brand  km_driven    fuel         owner  selling_price
0   Maruti     145500  Diesel   First Owner         450000
1    Skoda     120000  Diesel  Second Owner         370000
2    Honda     140000  Petrol   Third Owner         158000
3  Hyundai     127000  Diesel   First Owner         225000
4   Maruti     120000  Petrol   First Owner         130000
Unique Brands: 32

Brand Counts:
 brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG          

In [3]:
# 1. Standard One-Hot Encoding (5 original cols -> 3 + 4 + 5 = 12 cols)
df_ohe = pd.get_dummies(df, columns=['fuel', 'owner'])
print("Shape with all dummy columns:", df_ohe.shape)

# 2. K-1 One-Hot Encoding (drops 'CNG' from fuel and 'First Owner' from owner -> 10 cols)
df_ohe_k1 = pd.get_dummies(df, columns=['fuel', 'owner'], drop_first=True)
print("Shape with drop_first=True:", df_ohe_k1.shape)
print(df_ohe_k1.head())

Shape with all dummy columns: (8128, 12)
Shape with drop_first=True: (8128, 10)
     brand  km_driven  selling_price  fuel_Diesel  fuel_LPG  fuel_Petrol  \
0   Maruti     145500         450000         True     False        False   
1    Skoda     120000         370000         True     False        False   
2    Honda     140000         158000        False     False         True   
3  Hyundai     127000         225000         True     False        False   
4   Maruti     120000         130000        False     False         True   

   owner_Fourth & Above Owner  owner_Second Owner  owner_Test Drive Car  \
0                       False               False                 False   
1                       False                True                 False   
2                       False               False                 False   
3                       False               False                 False   
4                       False               False                 False   

   owner_Thi

In Real World we use sklearn instead of pandas

In [4]:

# We use sklearn insted of pandas because pandas doesnt store the encoder object for future use. We will use sklearn's OneHotEncoder to fit and transform the data, and then we can save the encoder for future use.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# 1. Train-Test Split (first 4 columns as X, selling_price as y)
X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, 0:4],
    df.iloc[:, -1],
    test_size=0.2,
    random_state=2
)

# 2. Initialize OneHotEncoder
# drop='first' removes the first category of each feature to prevent multicollinearity
# sparse_output=False (or sparse=False in older sklearn) returns a dense NumPy array directly
# dtype=np.int32 converts floats to integers
ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int32)

# 3. Fit and transform only 'fuel' and 'owner' columns
X_train_new = ohe.fit_transform(X_train[['fuel', 'owner']])
X_test_new = ohe.transform(X_test[['fuel', 'owner']])

print("Encoded feature shape (4-1 + 5-1 = 7 columns):", X_train_new.shape)

# 4. Horizontally stack ('brand', 'km_driven') with the newly encoded ('fuel', 'owner') array
X_train_combined = np.hstack((X_train[['brand', 'km_driven']].values, X_train_new))
print("Combined X_train shape (2 + 7 = 9 columns):", X_train_combined.shape)

Encoded feature shape (4-1 + 5-1 = 7 columns): (6502, 7)
Combined X_train shape (2 + 7 = 9 columns): (6502, 9)


In [5]:
# Count cars per brand and set threshold to 100
counts = df['brand'].value_counts()
threshold = 100

# Extract brand names that appear <= 100 times
repl = counts[counts <= threshold].index

# Replace infrequent brands with 'uncommon' and apply get_dummies
df_brand_ohe = pd.get_dummies(df['brand'].replace(repl, 'uncommon'), dtype=int)

print("Reduced Brand Dummy Columns:", df_brand_ohe.shape[1]) # 12 columns instead of 32
print(df_brand_ohe.sample(5))

Reduced Brand Dummy Columns: 13
      BMW  Chevrolet  Ford  Honda  Hyundai  Mahindra  Maruti  Renault  Skoda  \
4618    0          0     0      0        0         0       1        0      0   
8049    0          0     0      1        0         0       0        0      0   
3885    0          0     0      0        1         0       0        0      0   
1161    0          0     0      0        0         0       1        0      0   
650     0          0     0      0        0         0       0        0      0   

      Tata  Toyota  Volkswagen  uncommon  
4618     0       0           0         0  
8049     0       0           0         0  
3885     0       0           0         0  
1161     0       0           0         0  
650      0       0           0         1  
